In [ ]:
import h5py
import numpy as np
import umap
import plotly.graph_objs as go

# Ruta al archivo
ruta = "EmbeddingsRawDS[716]&AmyAVariants.h5"

# Definir categorías generales
categorias = {
    "Hidrolasas Archaea": ["Ha"],
    "Hidrolasas Bacterianas": ["Hb"],
    "Hidrolasas Eucariotas": ["He"],
    "Transferasas Archaea": ["Ta"],
    "Transferasas Bacterianas": ["Tb"],
    "Transferasas Eucariotas": ["Te"],
    "AmyA": ["AmyA"]
}

# Inicializar almacenamiento dinámico
grupos = {}
claves = []
etiquetas = []  # Etiquetas por categoría

# Abrir el archivo y recorrer todas las claves
with h5py.File(ruta, 'r') as archivo:
    for clave in archivo.keys():
        if isinstance(archivo[clave], h5py.Dataset):  # Filtrar solo datasets
            grupos[clave] = archivo[clave][:]
            claves.append(clave)

            # Asignar la categoría correcta según los prefijos
            for categoria, subgrupos in categorias.items():
                if any(clave.startswith(subgrupo) for subgrupo in subgrupos):
                    etiquetas.append(categoria)
                    break

# Construir X e y
X = np.vstack([np.array(grupos[clave]) for clave in grupos])

print(f"Total de embeddings cargados: {len(etiquetas)}")
print("Forma de X:", X.shape)

# Aplicar UMAP en 3D
reductor = umap.UMAP(
    n_components=3,
    metric='cosine',
    n_neighbors=60,
    min_dist=0.1,
    random_state=42
)
X_3d = reductor.fit_transform(X)

# Asignar colores únicos a las categorías
# Update colores dictionary to include all categories from 'categorias'
colores = {
    "Hidrolasas Archaea": "#000080",  # Azul navy
    "Hidrolasas Bacterianas": "#DC143C",  # Rojo intenso
    "Hidrolasas Eucariotas": "#FFFF00",  # Amarillo
    "Transferasas Archaea": "#FFA500",  # Naranja
    "Transferasas Bacterianas": "#006400",  # Verde
    "Transferasas Eucariotas": "#800080",  # Morado
    "AmyA": "#000000"  # Black
}

# Crear lista de trazas (una por categoría)
trazas = []
for categoria in categorias.keys():
    indices = [i for i, etiqueta in enumerate(etiquetas) if etiqueta == categoria]
    textos = [claves[i] for i in indices]  # Nombres para hover text
    traza = go.Scatter3d(
        x=X_3d[indices, 0],
        y=X_3d[indices, 1],
        z=X_3d[indices, 2],
        mode='markers',
        name=categoria,
        text=textos,
        hoverinfo='text+name',
        marker=dict(
            size=5,
            color=colores[categoria],
            opacity=0.7,
            line=dict(width=0.5, color='black')
        )
    )
    trazas.append(traza)

# Crear la figura
fig = go.Figure(data=trazas)

fig.update_layout(
    title='Proyección UMAP 3D con Agrupación por Categorías',
    scene=dict(
        xaxis_title='UMAP 1',
        yaxis_title='UMAP 2',
        zaxis_title='UMAP 3'
    ),
    legend_title='Categorías Enzimáticas',
    margin=dict(l=0, r=0, b=0, t=30),
    width=900,
    height=700
)

# Mostrar figura
fig.show()

# Guardar la figura como archivo HTML
fig.write_html("UMAP3D_Categorized_Embeddings.html")

print("Visualización guardada como 'UMAP3D_Categorized_Embeddings.html'")

Total de embeddings cargados: 695
Forma de X: (702, 1280)


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

/usr/local/lib/python3.11/dist-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



Visualización guardada como 'UMAP3D_Categorized_Embeddings.html'


CODIGO PARA ANALIZAR 667 EMBEDDINGS MAS LAS MUTANTES DE AMYA

In [ ]:
# Instalar dependencias necesarias
!pip install -q kaleido

import h5py
import numpy as np
import umap
import plotly.graph_objs as go

# === ARCHIVOS ===
ruta_original = "/content/Embeddings667.h5"
ruta_nuevos = "/content/EmbeddingsAmyA&Mutantes[6].h5"

# === CATEGORÍAS Y COLORES ===
categorias = {
    "Hidrolasas Archaea": ["Ha"],
    "Hidrolasas Bacterianas": ["Hb"],
    "Hidrolasas Eucariotas": ["He"],
    "Transferasas Archaea": ["Ta"],
    "Transferasas Bacterianas": ["Tb"],
    "Transferasas Eucariotas": ["Te"],
    "AmyA": ["AmyA"]
}
colores = {
    "Hidrolasas Archaea": "#000080",
    "Hidrolasas Bacterianas": "#8B0000",
    "Hidrolasas Eucariotas": "#FFFF00",
    "Transferasas Archaea": "#8B4513",
    "Transferasas Bacterianas": "#006400",
    "Transferasas Eucariotas": "#4B0082",
    "AmyA": "#000000"
}

# === CARGAR EMBEDDINGS ORIGINALES ===
grupos = {}
claves = []
etiquetas = []
with h5py.File(ruta_original, 'r') as archivo:
    for clave in archivo.keys():
        if isinstance(archivo[clave], h5py.Dataset):
            grupos[clave] = archivo[clave][:]
            claves.append(clave)
            for categoria, subgrupos in categorias.items():
                if any(clave.startswith(sub) for sub in subgrupos):
                    etiquetas.append(categoria)
                    break

X = np.vstack([grupos[clave] for clave in claves])
print(f"Total de embeddings cargados: {len(etiquetas)}")
print("Forma de X:", X.shape)

# === UMAP ORIGINAL ===
reductor = umap.UMAP(
    n_components=2,
    metric='cosine',
    n_neighbors=30,
    min_dist=0.001,
    random_state=42
)
X_2d = reductor.fit_transform(X)

# === TRACES ORIGINALES ===
trazas = []
for categoria in categorias.keys():
    indices = [i for i, etiqueta in enumerate(etiquetas) if etiqueta == categoria]
    textos = [claves[i] for i in indices]
    traza = go.Scatter(
        x=X_2d[indices, 0],
        y=X_2d[indices, 1],
        mode='markers',
        name=categoria,
        text=textos,
        hoverinfo='text+name',
        marker=dict(
            size=6,
            color=colores[categoria],
            opacity=0.75,
            line=dict(width=0.5, color='black')
        )
    )
    trazas.append(traza)

# === CARGAR NUEVOS EMBEDDINGS (Mutantes AmyA) ===
grupos_nuevos = {}
claves_nuevas = []
with h5py.File(ruta_nuevos, 'r') as archivo_nuevo:
    for clave in archivo_nuevo.keys():
        if isinstance(archivo_nuevo[clave], h5py.Dataset):
            grupos_nuevos[clave] = archivo_nuevo[clave][:]
            claves_nuevas.append(clave)

X_nuevos = np.vstack([grupos_nuevos[clave] for clave in claves_nuevas])
assert X.shape[1] == X_nuevos.shape[1], "Las dimensiones no coinciden."

# === UMAP TRANSFORM PARA NUEVOS ===
X_nuevos_2d = reductor.transform(X_nuevos)

# === TRACES NUEVOS ===
traza_mutantes = go.Scatter(
    x=X_nuevos_2d[:, 0],
    y=X_nuevos_2d[:, 1],
    mode='markers',
    name="A",
    text=claves_nuevas,
    hoverinfo='text+name',
    marker=dict(
        size=10,
        color='black',
        symbol='diamond',
        opacity=1.0,
        line=dict(width=1, color='white')
    )
)
trazas.append(traza_mutantes)

# === FIGURA FINAL ===
fig = go.Figure(data=trazas)
fig.update_layout(
    title='GlicosilHidrolasas de la familia 13',
    xaxis_title='UMAP 1',
    yaxis_title='UMAP 2',
    legend_title='Categorías Enzimáticas',
    margin=dict(l=0, r=0, b=0, t=30),
    width=800,
    height=700
)

# === GUARDAR Y MOSTRAR ===
fig.show()
fig.write_html("UMAP2D_Categorized_Embeddings_and_Mutants.html")
fig.write_image("UMAP2D_Categorized_Embeddings.png", format="png", scale=3)
fig.write_image("UMAP2D_Categorized_Embeddings.jpg", format="jpg", scale=3)
print("Visualización guardada como HTML, PNG y JPG.")


Total de embeddings cargados: 667
Forma de X: (667, 1280)


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.

/usr/local/lib/python3.11/dist-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning:

'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.



ValueError: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
